In [27]:
!pip install torchsummary

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [28]:
!pip install contextlib

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


ERROR: Could not find a version that satisfies the requirement contextlib (from versions: none)
ERROR: No matching distribution found for contextlib


In [1]:
from torchinfo import summary

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import joblib
import requests_cache
from sklearn.preprocessing import RobustScaler
from metpy.calc import wind_components
from metpy.units import units
from openmeteo_requests import Client
from retry_requests import retry
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, LSTM, Dense
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
import joblib
from math import radians, sin, cos, sqrt, atan2
import openmeteo_requests
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
amman = pd.read_csv('Datasets-Ready/amman_Ready.csv')  

In [4]:
def df_to_X_y(X_df, y_df, window_size=48, future=24):
    X, y = [], []
    for i in range(len(X_df) - window_size - future + 1):
        X.append(X_df.iloc[i:i+window_size].values)
        y.append(y_df.iloc[i+window_size:i+window_size+future].values.flatten())
    return np.array(X), np.array(y)

In [5]:
df = amman
amman.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210384 entries, 0 to 210383
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Unnamed: 0            210384 non-null  object 
 1   Air Dew Point         210384 non-null  float64
 2   Air Temperature (OC)  210384 non-null  float64
 3   Humidity %            210384 non-null  float64
 4   Atmospheric Pressure  210384 non-null  float64
 5   Liquid Precipitation  210384 non-null  float64
 6   Cloud Cover %         210384 non-null  float64
 7   Wind_U                210384 non-null  float64
 8   Wind_V                210384 non-null  float64
dtypes: float64(8), object(1)
memory usage: 14.4+ MB


In [6]:

df.drop(['Liquid Precipitation'], axis=1, inplace=True)
df['time'] = pd.to_datetime(df['Unnamed: 0'])
df.drop(columns=['Unnamed: 0'], inplace=True)

df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month

    # Cyclical encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

input_features = ['Air Dew Point', 'Air Temperature (OC)', 'Humidity %',
                      'Atmospheric Pressure', 'Wind_U', 'Wind_V']
target_feature = 'Cloud Cover %'

  
split_index = int(len(df) * 0.95)
train_df = df.iloc[:split_index].copy()
val_df = df.iloc[split_index:].copy()

    # Scalers
input_scaler = RobustScaler()
input_scaler.fit(train_df[input_features])
train_df[input_features] = input_scaler.transform(train_df[input_features])
val_df[input_features] = input_scaler.transform(val_df[input_features])

target_scaler = RobustScaler()
target_scaler.fit(train_df[[target_feature]])
train_df[target_feature] = target_scaler.transform(train_df[[target_feature]])
val_df[target_feature] = target_scaler.transform(val_df[[target_feature]])

    # Save scalers

# joblib.dump(input_scaler, input_scaler_path)
# joblib.dump(target_scaler, target_scaler_path)

    # Apply window 
X_train, y_train = df_to_X_y(train_df[input_features], train_df[[target_feature]])
X_val, y_val = df_to_X_y(val_df[input_features], val_df[[target_feature]])

    ############## Model Definition and Training ##############
model = Sequential([
        InputLayer(shape=(48, 6)),
        LSTM(64, return_sequences=True),
        LSTM(128),
        Dense(64, activation='relu'),
        Dense(24)  
])

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 48, 64)              │          18,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 24)                  │           1,560 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 126,808 (495.34 KB)

 Trainable params: 126,808 (495.34 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
class rain_dataset(Dataset):
    def __init__(self, df, seq_len=48, output_len=12):
        self.seq_len = seq_len
        self.output_len = output_len
        self.features = df[features].values.astype(np.float32)
        self.timestamps = df['time'].reset_index(drop=True)
        self.precip = df['Liquid Precipitation'].reset_index(drop=True).values
        self.X, self.y = [], []

        #sliding window
        for i in range(len(df) - seq_len - output_len):
            x_window = self.features[i:i+seq_len]
            y_hat = self.precip[i+seq_len:i+seq_len+output_len]

            start_time = self.timestamps[i]
            end_time = self.timestamps[i + seq_len + output_len - 1]
            expected_hours = seq_len + output_len - 1
            if (end_time - start_time).total_seconds() / 3600 != expected_hours:
                continue  

            y = float(np.sum(y_hat) > 0) 
            self.X.append(x_window)
            self.y.append(y)

        self.X = np.array(self.X, dtype=np.float32)
        self.y = np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

In [14]:
class RainLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(RainLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  
        )

    def forward(self, x):
        _, (hn, _) = self.lstm(x) 
        return self.fc(hn[-1])  

In [15]:
features = [
    'Air Dew Point', 'Air Temperature (OC)', 'Humidity %',
    'Atmospheric Pressure', 'Cloud Cover %', 'Wind_U', 'Wind_V',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df['time'] = pd.to_datetime(df['Unnamed: 0'])
df.drop(columns=['Unnamed: 0'], inplace=True)

df['hour'] = df['time'].dt.hour
df['day_yr'] = df['time'].dt.dayofyear
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_sin'] = np.sin(2 * np.pi * df['day_yr'] / 365)
df['day_cos'] = np.cos(2 * np.pi * df['day_yr'] / 365)

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
    


df['date'] = df['time'].dt.date
rain_per_day = df.groupby('date')['Liquid Precipitation'].sum()
rain_tomorrow = (rain_per_day.shift(-1) > 0).astype(int)
df['RainTomorrow'] = df['date'].map(rain_tomorrow)

df = df.dropna(subset=['RainTomorrow'])

full = rain_dataset(df)
    

train_len = int(len(full) * 0.95)
train_ds = torch.utils.data.Subset(full, range(train_len))
val_ds = torch.utils.data.Subset(full, range(train_len, len(full)))

model = RainLSTM(input_size=len(features)).to(device)

In [23]:
summary(model, input_size=(32, 48, len(features)))

Layer (type:depth-idx)                   Output Shape              Param #
RainLSTM                                 [32, 1]                   --
├─LSTM: 1-1                              [32, 48, 64]              52,992
├─Sequential: 1-2                        [32, 1]                   --
│    └─Linear: 2-1                       [32, 32]                  2,080
│    └─ReLU: 2-2                         [32, 32]                  --
│    └─Linear: 2-3                       [32, 1]                   33
│    └─Sigmoid: 2-4                      [32, 1]                   --
Total params: 55,105
Trainable params: 55,105
Non-trainable params: 0
Total mult-adds (M): 81.46
Input size (MB): 0.07
Forward/backward pass size (MB): 0.79
Params size (MB): 0.22
Estimated Total Size (MB): 1.08